In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
import os

# paths
AA_OUT = r"C:\Users\user\Downloads\GSE148375_clean"
EA_OUT = r"C:\Users\user\Downloads\GSE148812_clean"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

# reload manifest for gene annotation
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(lambda x: re.sub(r'_\d+$', '', x))
chr_lookup = manifest_df.set_index("core_name")["Chr"]
pos_lookup = manifest_df.set_index("core_name")["MapInfo"]

def get_chr(pid):
    return str(chr_lookup.get(re.sub(r'_\d+$', '', pid), "?"))

def get_pos(pid):
    val = pos_lookup.get(re.sub(r'_\d+$', '', pid), None)
    return int(val) if pd.notna(val) else None

# load shortlists
aa_shortlist = pd.read_csv(os.path.join(AA_OUT, "v2_shortlist_ld_pruned.csv"))
ea_shortlist = pd.read_csv(os.path.join(EA_OUT, "v2_shortlist_final.csv"))

print("AA shortlist:", len(aa_shortlist), "SNPs")
print("EA shortlist:", len(ea_shortlist), "SNPs")

# AA network SNPs connected to smoking_status (from our results)
aa_smoking_snps = [
    "exm100944-0_B_R_1921482882",
    "exm2224359-0_B_F_1960100656",
    "exm1245580-0_B_F_2060131617",
    "exm1379120-0_T_F_1921625489",
    "exm1421650-0_T_F_1923324725",
    "exm236098-0_T_R_1918982506",
    "exm317474-0_B_F_1922239353",
    "exm418184-0_B_R_1923021066",
    "exm609218-0_B_F_1918575531",
    "exm759751-0_T_R_1922412944",
    "exm777999-0_B_F_1922426645",
    "exm782555-0_B_R_1922302526"
]

# EA network SNPs connected to smoking_status
ea_smoking_snps = [
    "exm935491-0_T_R_1918372056",
    "exm793377-0_T_R_1922407882"
]

print("\nAA SNPs → smoking_status:", len(aa_smoking_snps))
print("EA SNPs → smoking_status:", len(ea_smoking_snps))

AA shortlist: 56 SNPs
EA shortlist: 18 SNPs

AA SNPs → smoking_status: 12
EA SNPs → smoking_status: 2


In [2]:
# overlap analysis: compare AA and EA shortlists by genomic position
# a SNP is "shared" if it appears in both cohorts within 100kb of each other on the same chromosome

window_bp = 100_000

def get_snp_positions(probe_ids):
    positions = []
    for pid in probe_ids:
        chrom = get_chr(pid)
        pos = get_pos(pid)
        if pos:
            positions.append({"probe_id": pid, "Chr": chrom, "pos": pos})
    return pd.DataFrame(positions)

aa_pos = get_snp_positions(aa_shortlist["probe_id"].tolist())
ea_pos = get_snp_positions(ea_shortlist["probe_id"].tolist())

print("=== SHORTLIST OVERLAP (56 AA vs 18 EA SNPs) ===\n")
shared_shortlist = []
for _, row_aa in aa_pos.iterrows():
    for _, row_ea in ea_pos.iterrows():
        if row_aa["Chr"] == row_ea["Chr"]:
            dist = abs(row_aa["pos"] - row_ea["pos"])
            if dist <= window_bp:
                shared_shortlist.append({
                    "AA_probe": row_aa["probe_id"],
                    "EA_probe": row_ea["probe_id"],
                    "Chr": row_aa["Chr"],
                    "AA_pos": row_aa["pos"],
                    "EA_pos": row_ea["pos"],
                    "distance_bp": dist
                })

shared_df = pd.DataFrame(shared_shortlist)
print(f"Shared SNP regions (within 100kb): {len(shared_df)}")
if len(shared_df) > 0:
    print(shared_df.to_string())

# now compare specifically the smoking_status-connected SNPs
print("\n=== SMOKING_STATUS EDGE OVERLAP ===\n")
aa_smoking_pos = get_snp_positions(aa_smoking_snps)
ea_smoking_pos = get_snp_positions(ea_smoking_snps)

shared_smoking = []
for _, row_aa in aa_smoking_pos.iterrows():
    for _, row_ea in ea_smoking_pos.iterrows():
        if row_aa["Chr"] == row_ea["Chr"]:
            dist = abs(row_aa["pos"] - row_ea["pos"])
            if dist <= window_bp:
                shared_smoking.append({
                    "AA_probe": row_aa["probe_id"],
                    "EA_probe": row_ea["probe_id"],
                    "Chr": row_aa["Chr"],
                    "distance_bp": dist
                })

print(f"Shared smoking_status edges (within 100kb): {len(shared_smoking)}")
if len(shared_smoking) > 0:
    print(pd.DataFrame(shared_smoking).to_string())
else:
    print("No direct overlap in smoking_status-connected SNPs between cohorts")

# chromosome distribution comparison
print("\n=== CHROMOSOME DISTRIBUTION ===")
print("\nAA smoking SNPs by chromosome:")
print(aa_smoking_pos["Chr"].value_counts().sort_index())
print("\nEA smoking SNPs by chromosome:")
print(ea_smoking_pos["Chr"].value_counts().sort_index())

=== SHORTLIST OVERLAP (56 AA vs 18 EA SNPs) ===

Shared SNP regions (within 100kb): 0

=== SMOKING_STATUS EDGE OVERLAP ===

Shared smoking_status edges (within 100kb): 0
No direct overlap in smoking_status-connected SNPs between cohorts

=== CHROMOSOME DISTRIBUTION ===

AA smoking SNPs by chromosome:
Chr
1     1
16    2
18    1
19    1
2     1
3     1
4     1
7     1
9     3
Name: count, dtype: int64

EA smoking SNPs by chromosome:
Chr
11    1
9     1
Name: count, dtype: int64


In [3]:
print("Chr9 SNPs in AA network:")
for pid in aa_smoking_snps:
    if get_chr(pid) == "9":
        print(f"  {pid}: pos={get_pos(pid):,}")

print("\nChr9 SNPs in EA network:")
for pid in ea_smoking_snps:
    if get_chr(pid) == "9":
        print(f"  {pid}: pos={get_pos(pid):,}")

# also check broader window (1Mb) for any overlap
print("\n=== BROADER OVERLAP CHECK (1Mb window) ===")
window_1mb = 1_000_000
shared_1mb = []
for _, row_aa in aa_smoking_pos.iterrows():
    for _, row_ea in ea_smoking_pos.iterrows():
        if row_aa["Chr"] == row_ea["Chr"]:
            dist = abs(row_aa["pos"] - row_ea["pos"])
            if dist <= window_1mb:
                shared_1mb.append({
                    "AA_probe": row_aa["probe_id"],
                    "EA_probe": row_ea["probe_id"],
                    "Chr": row_aa["Chr"],
                    "AA_pos": row_aa["pos"],
                    "EA_pos": row_ea["pos"],
                    "distance_bp": dist
                })

if shared_1mb:
    print(pd.DataFrame(shared_1mb).to_string())
else:
    print("Still no overlap at 1Mb window")

Chr9 SNPs in AA network:
  exm759751-0_T_R_1922412944: pos=90,584,110
  exm777999-0_B_F_1922426645: pos=123,914,765
  exm782555-0_B_R_1922302526: pos=130,206,472

Chr9 SNPs in EA network:
  exm793377-0_T_R_1922407882: pos=136,291,361

=== BROADER OVERLAP CHECK (1Mb window) ===
Still no overlap at 1Mb window


In [5]:
import pandas as pd
import numpy as np
import re
import os

AA_OUT = r"C:\Users\user\Downloads\GSE148375_clean"
EA_OUT = r"C:\Users\user\Downloads\GSE148812_clean"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(lambda x: re.sub(r'_\d+$', '', x))
chr_lookup = manifest_df.set_index("core_name")["Chr"]
pos_lookup_m = manifest_df.set_index("core_name")["MapInfo"]

def get_chr(pid):
    return str(chr_lookup.get(re.sub(r'_\d+$', '', pid), "?"))

def get_pos(pid):
    val = pos_lookup_m.get(re.sub(r'_\d+$', '', pid), None)
    return int(val) if pd.notna(val) else None

# load both shortlists from saved files
aa_shortlist = pd.read_csv(os.path.join(AA_OUT, "v2_shortlist_ld_pruned.csv"))
ea_shortlist = pd.read_csv(os.path.join(EA_OUT, "v2_shortlist_ld_pruned_v2.csv"))

print(f"AA shortlist: {len(aa_shortlist)} SNPs")
print(f"EA shortlist: {len(ea_shortlist)} SNPs")

def find_overlap(aa_ids, ea_ids, window_bp):
    aa_pos = [(pid, get_chr(pid), get_pos(pid)) for pid in aa_ids]
    ea_pos = [(pid, get_chr(pid), get_pos(pid)) for pid in ea_ids]
    shared = []
    for aa_pid, aa_chr, aa_p in aa_pos:
        for ea_pid, ea_chr, ea_p in ea_pos:
            if aa_chr == ea_chr and aa_p and ea_p:
                dist = abs(aa_p - ea_p)
                if dist <= window_bp:
                    shared.append({
                        "AA_probe": aa_pid,
                        "EA_probe": ea_pid,
                        "Chr": aa_chr,
                        "AA_pos": aa_p,
                        "EA_pos": ea_p,
                        "distance_bp": dist
                    })
    return pd.DataFrame(shared)

print("\n=== SHORTLIST OVERLAP (matched parameters) ===")
for window in [100_000, 500_000, 1_000_000]:
    shared = find_overlap(
        aa_shortlist["probe_id"].tolist(),
        ea_shortlist["probe_id"].tolist(),
        window
    )
    print(f"Within {window//1000}kb: {len(shared)} shared regions")
    if len(shared) > 0 and window == 1_000_000:
        print(shared[["Chr", "AA_pos", "EA_pos", "distance_bp"]].to_string())

AA shortlist: 56 SNPs
EA shortlist: 52 SNPs

=== SHORTLIST OVERLAP (matched parameters) ===
Within 100kb: 0 shared regions
Within 500kb: 3 shared regions
Within 1000kb: 5 shared regions
  Chr     AA_pos     EA_pos  distance_bp
0   1  206972307  207195568       223261
1   1  207875175  207195568       679607
2  19   45322316   44501481       820835
3   4    1818625    1656776       161849
4   6   30618906   31068026       449120


In [6]:
import requests
import time

shared_1mb = find_overlap(
    aa_shortlist["probe_id"].tolist(),
    ea_shortlist["probe_id"].tolist(),
    1_000_000
)

def get_gene_ensembl(chrom, pos, window=50000):
    url = f"https://grch37.rest.ensembl.org/overlap/region/human/{chrom}:{pos-window}-{pos+window}"
    params = {"feature": "gene", "content-type": "application/json"}
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            return "?"
        genes = r.json()
        if not genes:
            return "intergenic"
        coding = [g for g in genes if g.get("biotype") == "protein_coding"]
        targets = coding if coding else genes
        names = list(set(g.get("external_name", "?") for g in targets))
        return " | ".join(names[:3])
    except:
        return "error"

print("=== SHARED GENOMIC REGIONS — GENE ANNOTATION ===\n")
for _, row in shared_1mb.iterrows():
    mid_pos = (row["AA_pos"] + row["EA_pos"]) // 2
    gene_aa = get_gene_ensembl(row["Chr"], row["AA_pos"])
    gene_ea = get_gene_ensembl(row["Chr"], row["EA_pos"])
    print(f"Chr{row['Chr']} | Distance: {row['distance_bp']:,} bp")
    print(f"  AA pos {row['AA_pos']:,} → {gene_aa}")
    print(f"  EA pos {row['EA_pos']:,} → {gene_ea}")
    print()
    time.sleep(0.5)

=== SHARED GENOMIC REGIONS — GENE ANNOTATION ===

Chr1 | Distance: 223,261 bp
  AA pos 206,972,307 → IL10 | IL19
  EA pos 207,195,568 → PFKFB2 | YOD1 | C1orf116

Chr1 | Distance: 679,607 bp
  AA pos 207,875,175 → CR1L
  EA pos 207,195,568 → PFKFB2 | YOD1 | C1orf116

Chr19 | Distance: 820,835 bp
  AA pos 45,322,316 → BCAM | PVRL2 | CBLC
  EA pos 44,501,481 → ZNF155 | ZNF221 | ZNF222

Chr4 | Distance: 161,849 bp
  AA pos 1,818,625 → LETM1 | FGFR3
  EA pos 1,656,776 → SLBP | FAM53A

Chr6 | Distance: 449,120 bp
  AA pos 30,618,906 → PPP1R18 | MDC1 | DHX16
  EA pos 31,068,026 → CDSN | PSORS1C1 | PSORS1C2



In [7]:
import pandas as pd
import re
import os

MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(lambda x: re.sub(r'_\d+$', '', x))
chr_lookup = manifest_df.set_index("core_name")["Chr"]
pos_lookup_m = manifest_df.set_index("core_name")["MapInfo"]

def get_chr(pid):
    return str(chr_lookup.get(re.sub(r'_\d+$', '', pid), "?"))

def get_pos(pid):
    val = pos_lookup_m.get(re.sub(r'_\d+$', '', pid), None)
    return int(val) if pd.notna(val) else None

# AA smoking SNPs (from our v2 network)
aa_smoking_snps = [
    "exm100944-0_B_R_1921482882",
    "exm2224359-0_B_F_1960100656",
    "exm1245580-0_B_F_2060131617",
    "exm1379120-0_T_F_1921625489",
    "exm1421650-0_T_F_1923324725",
    "exm236098-0_T_R_1918982506",
    "exm317474-0_B_F_1922239353",
    "exm418184-0_B_R_1923021066",
    "exm609218-0_B_F_1918575531",
    "exm759751-0_T_R_1922412944",
    "exm777999-0_B_F_1922426645",
    "exm782555-0_B_R_1922302526"
]

# EA smoking SNPs (v2 network)
ea_smoking_snps_v2 = [
    "exm71047-0_B_R_1921357564",
    "exm834716-0_B_R_1920965043",
    "exm850504-0_B_R_1921041917",
    "exm935491-0_T_R_1918372056",
    "exm623475-0_B_F_1918444599"
]

print("AA SNPs → smoking_status:", len(aa_smoking_snps))
print("EA SNPs → smoking_status:", len(ea_smoking_snps_v2))

# position-based overlap at multiple windows
def find_overlap(aa_ids, ea_ids, window_bp):
    shared = []
    for aa_pid in aa_ids:
        aa_chr, aa_p = get_chr(aa_pid), get_pos(aa_pid)
        for ea_pid in ea_ids:
            ea_chr, ea_p = get_chr(ea_pid), get_pos(ea_pid)
            if aa_chr == ea_chr and aa_p and ea_p:
                dist = abs(aa_p - ea_p)
                if dist <= window_bp:
                    shared.append({
                        "AA_probe": aa_pid,
                        "EA_probe": ea_pid,
                        "Chr": aa_chr,
                        "AA_pos": aa_p,
                        "EA_pos": ea_p,
                        "distance_bp": dist
                    })
    return pd.DataFrame(shared)

print("\n=== SMOKING_STATUS EDGE OVERLAP ===")
for window in [100_000, 500_000, 1_000_000]:
    shared = find_overlap(aa_smoking_snps, ea_smoking_snps_v2, window)
    print(f"Within {window//1000}kb: {len(shared)} shared signals")
    if len(shared) > 0:
        for _, row in shared.iterrows():
            print(f"  Chr{row['Chr']}: AA={row['AA_pos']:,} | EA={row['EA_pos']:,} | dist={row['distance_bp']:,}bp")

AA SNPs → smoking_status: 12
EA SNPs → smoking_status: 5

=== SMOKING_STATUS EDGE OVERLAP ===
Within 100kb: 0 shared signals
Within 500kb: 0 shared signals
Within 1000kb: 0 shared signals


In [8]:
print("AA smoking SNPs — chromosomes and positions:")
for pid in aa_smoking_snps:
    print(f"  Chr{get_chr(pid)}:{get_pos(pid):,}  {pid[:35]}")

print("\nEA smoking SNPs — chromosomes and positions:")
for pid in ea_smoking_snps_v2:
    print(f"  Chr{get_chr(pid)}:{get_pos(pid):,}  {pid[:35]}")

# check chromosome overlap at least
aa_chroms = set(get_chr(p) for p in aa_smoking_snps)
ea_chroms = set(get_chr(p) for p in ea_smoking_snps_v2)
print(f"\nAA chromosomes: {sorted(aa_chroms)}")
print(f"EA chromosomes: {sorted(ea_chroms)}")
print(f"Shared chromosomes: {sorted(aa_chroms & ea_chroms)}")

AA smoking SNPs — chromosomes and positions:
  Chr1:152,329,460  exm100944-0_B_R_1921482882
  Chr16:20,435,321  exm2224359-0_B_F_1960100656
  Chr16:58,320,066  exm1245580-0_B_F_2060131617
  Chr18:21,530,071  exm1379120-0_T_F_1921625489
  Chr19:9,074,017  exm1421650-0_T_F_1923324725
  Chr2:160,889,476  exm236098-0_T_R_1918982506
  Chr3:50,312,822  exm317474-0_B_F_1922239353
  Chr4:110,394,162  exm418184-0_B_R_1923021066
  Chr7:22,985,282  exm609218-0_B_F_1918575531
  Chr9:90,584,110  exm759751-0_T_R_1922412944
  Chr9:123,914,765  exm777999-0_B_F_1922426645
  Chr9:130,206,472  exm782555-0_B_R_1922302526

EA smoking SNPs — chromosomes and positions:
  Chr1:85,020,695  exm71047-0_B_R_1921357564
  Chr10:75,523,634  exm834716-0_B_R_1920965043
  Chr10:102,824,292  exm850504-0_B_R_1921041917
  Chr11:68,562,288  exm935491-0_T_R_1918372056
  Chr7:64,451,709  exm623475-0_B_F_1918444599

AA chromosomes: ['1', '16', '18', '19', '2', '3', '4', '7', '9']
EA chromosomes: ['1', '10', '11', '7']
Shared 